# Topic 39 — Hyperparameter Tuning
### Theory → GridSearchCV → RandomizedSearchCV → applying it to your Topic 25 text pipeline → Optuna preview.

Recall (Topic 5): **parameters** are learned FROM data (weights, biases). **Hyperparameters** are
choices YOU make before training (e.g. `C` in SVM, `n_estimators` in Random Forest, `max_depth` in
a tree, `ngram_range` in TfidfVectorizer). Hyperparameter tuning = systematically searching over
combinations of these to find the best-performing configuration, using cross-validation
(Topic 6) to evaluate each candidate fairly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform, randint

rng = np.random.default_rng(0)

## 1. Search space — defining what to try

A **search space** (a.k.a. parameter grid) specifies which hyperparameters to vary and what values
to try for each. For a `Pipeline` (Topic 17), you reference a step's parameter with
`stepname__paramname` (double underscore).

In [ ]:
X, y = make_classification(n_samples=300, n_features=10, n_informative=6, random_state=42)

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", 0.01, 0.1, 1],
}
# GridSearchCV will try EVERY combination: 5 * 2 * 4 = 40 combinations (kernel="linear" ignores gamma,
# but sklearn still tries every listed value regardless -- some waste is normal with grid search)
print("total combinations in this grid:", 5 * 2 * 4)

## 2. Grid search — exhaustive search over every combination

`GridSearchCV` trains and cross-validates a model for EVERY combination in the grid, then reports
the best one. Guaranteed to find the best combination WITHIN the grid you defined, but the cost
grows multiplicatively with the number of hyperparameters and values — can get slow fast.

In [ ]:
grid_search = GridSearchCV(
    SVC(), param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,   # use all CPU cores
)
grid_search.fit(X, y)

print("best parameters:", grid_search.best_params_)
print("best cross-validated F1:", grid_search.best_score_)

results_df = pd.DataFrame(grid_search.cv_results_)
print("\ntop 5 combinations:")
print(results_df.sort_values("mean_test_score", ascending=False)[
    ["param_C", "param_kernel", "param_gamma", "mean_test_score"]
].head())

## 3. Random search — sampling instead of exhausting

`RandomizedSearchCV` samples a fixed NUMBER of random combinations from the search space (which can
include continuous DISTRIBUTIONS, not just fixed lists), rather than trying every single one.
Often finds a nearly-as-good result as grid search in a fraction of the time, especially when the
search space is large or has many hyperparameters.

In [ ]:
param_distributions = {
    "n_estimators": randint(50, 300),          # random integer in [50, 300)
    "max_depth": randint(2, 20),
    "min_samples_split": randint(2, 20),
    "max_features": uniform(0.3, 0.7),          # random float in [0.3, 1.0)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), param_distributions,
    n_iter=30,   # only try 30 random combinations, not all possible ones
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1", n_jobs=-1, random_state=42,
)
random_search.fit(X, y)

print("best parameters:", random_search.best_params_)
print("best cross-validated F1:", random_search.best_score_)

## 4. Grid search vs random search — visualizing search coverage

For a large search space, random search explores a WIDER variety of combinations for the same
number of trials, which often matters more than exhaustively covering a small grid.

In [ ]:
# Grid: fixed points on a lattice
grid_points_x = [1, 2, 3, 4, 5]
grid_points_y = [1, 2, 3, 4, 5]
grid_combos = [(x, y) for x in grid_points_x for y in grid_points_y]

# Random: scattered points anywhere in the same range
random_combos = list(zip(rng.uniform(1, 5, 25), rng.uniform(1, 5, 25)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].scatter(*zip(*grid_combos), color="blue")
axes[0].set_title("Grid search (25 fixed lattice points)")
axes[1].scatter(*zip(*random_combos), color="orange")
axes[1].set_title("Random search (25 randomly scattered points)")
for ax in axes:
    ax.set_xlabel("hyperparameter A"); ax.set_ylabel("hyperparameter B")
plt.tight_layout()
plt.show()
# If only ONE hyperparameter actually matters, grid search wastes many trials repeating the
# same value of the important one -- random search naturally avoids this.

## 5. Tuning your actual Topic 25 text classification pipeline

Applying this directly to the TF-IDF + classifier pipeline from Topic 25 — tuning BOTH the
vectorizer's hyperparameters (`ngram_range`, `min_df`) and the classifier's (`C`), all in ONE search.

In [ ]:
bullying_examples = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "you should just disappear", "you are pathetic",
    "everyone thinks you're an idiot", "just go away nobody likes you",
    "you're so ugly and useless", "why do you even exist",
    "you deserve to be alone", "stop talking you sound stupid",
]
not_bullying_examples = [
    "great job today team", "have a wonderful day", "nice work everyone",
    "thanks for your help", "well done on the project", "excellent effort today",
    "looking forward to the weekend", "congratulations on your achievement",
    "the weather is nice today", "let's grab coffee sometime",
    "i really appreciate your feedback", "the meeting went smoothly",
]
texts = bullying_examples + not_bullying_examples
y_text = np.array([1]*len(bullying_examples) + [0]*len(not_bullying_examples))

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

text_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.1, 1, 10],
}

text_grid_search = GridSearchCV(
    text_pipeline, text_param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),   # small cv due to tiny dataset
    scoring="f1",
)
text_grid_search.fit(texts, y_text)

print("best text pipeline params:", text_grid_search.best_params_)
print("best cross-validated F1:", text_grid_search.best_score_)

## 6. Important: always evaluate the FINAL choice on a held-out test set

Cross-validation during search reuses the same data repeatedly to pick hyperparameters — that
selection process itself can slightly overfit to the validation folds. Best practice: hold out a
separate final test set (Topic 6's "Train -> Validation -> Test" principle) never touched during
the search, and report performance on THAT for your paper's final numbers.

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_final_test, y_temp, y_final_test = train_test_split(
    texts, y_text, test_size=0.25, random_state=42, stratify=y_text
)

search_on_temp = GridSearchCV(text_pipeline, text_param_grid, cv=3, scoring="f1")
search_on_temp.fit(X_temp, y_temp)

final_test_score = search_on_temp.score(X_final_test, y_final_test)
print("best params found using only X_temp:", search_on_temp.best_params_)
print("honest final F1 on NEVER-touched test set:", final_test_score)

## 7. Optuna — a brief mention

`Optuna` is a more advanced hyperparameter optimization library that uses smarter search
strategies (e.g. Bayesian optimization — using results of PAST trials to intelligently choose the
NEXT trial to try, rather than randomly or exhaustively). Worth reaching for once grid/random
search feels too slow or you're tuning many hyperparameters at once — not necessary to start with.

In [ ]:
# pip install optuna -- example sketch, not run here:
#
# import optuna
# def objective(trial):
#     C = trial.suggest_float("C", 0.01, 100, log=True)
#     ngram_max = trial.suggest_int("ngram_max", 1, 3)
#     pipeline = Pipeline([
#         ("tfidf", TfidfVectorizer(ngram_range=(1, ngram_max))),
#         ("clf", LogisticRegression(C=C, class_weight="balanced", max_iter=1000)),
#     ])
#     scores = cross_val_score(pipeline, texts, y_text, cv=3, scoring="f1")
#     return scores.mean()
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50)
# print(study.best_params)
print("Optuna sketch above -- install with `pip install optuna` when your search space grows large.")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Expand text_param_grid to also tune tfidf__max_df -- how many total combinations does
#    that create, and does GridSearchCV noticeably slow down?
# 2. Run RandomizedSearchCV with n_iter=10 on the same text_param_grid (treating the grid's lists
#    as discrete choices) -- does it find a result close to GridSearchCV's, in less time?
# 3. Swap scoring="f1" for scoring="recall" in the SVM grid search from part 2 -- does the
#    'best' hyperparameter combination change? (Reconnect to Topic 9's precision/recall tradeoff.)
# 4. Once you have your real dataset, set aside a final test set FIRST (before any tuning),
#    tune on the remainder, then report the final test score -- exactly like part 6 above.

---
### Next up: **Topic 40 — Model Interpretation** (SHAP, LIME — important for research).

Say "next" when you're ready.